In [13]:
import mlflow
# import mlflow.sklearn
import optuna

from optuna.integration.mlflow import MLflowCallback
from optuna.pruners import MedianPruner

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold, cross_val_score

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    OneHotEncoder
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [14]:
X_train = pd.read_csv("../data/processed/raw_features/X_train.csv")
X_test = pd.read_csv("../data/processed/raw_features/X_test.csv")
y_train = pd.read_csv("../data/processed/target/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/target/y_test.csv").squeeze()

In [15]:
# =========================================================
# Columns
# =========================================================

dummyfy_columns = [
    'Card Type',
    'NumOfProducts',
    'Geography',
    'Gender'
]

norm_std_columns = [
    'Balance',
    'Point Earned',
    'CreditScore',
    'Age',
    'Tenure',
    'Satisfaction Score'
]

In [16]:
# =========================================================
# Config
# =========================================================

RANDOM_STATE = 42
N_SPLITS = 5

EXPERIMENT_NAME = "customer-churn-optuna"

engineer features.
find shap_values to determine most important features

set optuna and mlflow.

In [17]:
# =========================================================
# Build Pipeline
# =========================================================

def build_pipeline(trial):

    # -----------------------------
    # Preprocessing
    # -----------------------------

    scaler_name = trial.suggest_categorical(
        'scaler',
        ['std', 'minmax', 'robust']
    )

    encoder_name = trial.suggest_categorical(
        'encoder',
        ['drop_first', 'no_drop']
    )

    scalers = {
        'std': StandardScaler(),
        'minmax': MinMaxScaler(),
        'robust': RobustScaler()
    }

    scaler = scalers[scaler_name]

    encoder = OneHotEncoder(
        handle_unknown='ignore',
        drop='first' if encoder_name == 'drop_first' else None
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', encoder, dummyfy_columns),
            ('num', scaler, norm_std_columns)
        ]
    )

    # -----------------------------
    # Model Selection
    # -----------------------------

    model_name = trial.suggest_categorical(
        'model',
        ['lr', 'dt', 'rf']
    )

    if model_name == 'lr':

        model = LogisticRegression(
            C=trial.suggest_float(
                'lr_C',
                1e-3,
                10,
                log=True
            ),
            solver='lbfgs',
            max_iter=2000,
            random_state=RANDOM_STATE
        )

    elif model_name == 'dt':

        model = DecisionTreeClassifier(
            max_depth=trial.suggest_int(
                'dt_max_depth',
                2,
                20
            ),
            min_samples_split=trial.suggest_int(
                'dt_min_samples_split',
                2,
                20
            ),
            min_samples_leaf=trial.suggest_int(
                'dt_min_samples_leaf',
                1,
                10
            ),
            random_state=RANDOM_STATE
        )

    else:

        model = RandomForestClassifier(
            n_estimators=trial.suggest_int(
                'rf_n_estimators',
                100,
                500
            ),
            max_depth=trial.suggest_int(
                'rf_max_depth',
                2,
                20
            ),
            min_samples_split=trial.suggest_int(
                'rf_min_samples_split',
                2,
                20
            ),
            min_samples_leaf=trial.suggest_int(
                'rf_min_samples_leaf',
                1,
                10
            ),
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    # -----------------------------
    # Full Pipeline
    # -----------------------------

    pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('model', model)
    ])

    return pipeline

In [18]:
# =========================================================
# Objective Function
# =========================================================

def objective(trial):

    pipeline = build_pipeline(trial)

    cv = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        scoring='roc_auc',
        cv=cv,
        n_jobs=-1
    )

    mean_auc = np.mean(scores)

    # --------------------------------
    # Report intermediate value
    # (for pruning)
    # --------------------------------

    trial.report(mean_auc, step=0)

    if trial.should_prune():
        raise optuna.TrialPruned()

    return mean_auc

In [19]:
# =========================================================
# MLflow Setup
# =========================================================

mlflow.set_experiment(EXPERIMENT_NAME)

mlflc = MLflowCallback(
    tracking_uri=mlflow.get_tracking_uri(),
    metric_name="auc",
    create_experiment=False,
    mlflow_kwargs={"nested": True}
)


# =========================================================
# Study
# =========================================================

study = optuna.create_study(
    direction='maximize',
    pruner=MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=0
    )
)

/var/folders/bv/50x24wc545x5mclk_t88ryrc0000gn/T/ipykernel_74765/989846483.py:7: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflc = MLflowCallback(
[I 2026-05-08 11:08:25,441] A new study created in memory with name: no-name-3cfd5ce5-624f-4063-ae53-eca9cad7da82


In [26]:
# =========================================================
# Run Optimization
# =========================================================

with mlflow.start_run(run_name="optuna_search"):

    study.optimize(
        objective,
        n_trials=500,
        callbacks=[mlflc]
    )

    # --------------------------------
    # Best Metrics
    # --------------------------------

    mlflow.log_metric(
        "best_auc",
        study.best_value
    )

    # --------------------------------
    # Best Params
    # --------------------------------

    mlflow.log_params(study.best_params)

    # --------------------------------
    # Train Best Pipeline
    # --------------------------------

    best_pipeline = build_pipeline(study.best_trial)

    best_pipeline.fit(X_train, y_train)

    # --------------------------------
    # Save Best Model
    # --------------------------------

    mlflow.sklearn.log_model(
        sk_model=best_pipeline,
        name="best_model",
        serialization_format="skops"
    )

[I 2026-05-08 12:05:20,739] Trial 110 finished with value: 0.8474137311573848 and parameters: {'scaler': 'robust', 'encoder': 'drop_first', 'model': 'rf', 'rf_n_estimators': 133, 'rf_max_depth': 10, 'rf_min_samples_split': 19, 'rf_min_samples_leaf': 4}. Best is trial 101 with value: 0.8485263919579072.
[I 2026-05-08 12:05:22,298] Trial 111 pruned. 
[I 2026-05-08 12:05:24,147] Trial 112 finished with value: 0.848199002690494 and parameters: {'scaler': 'robust', 'encoder': 'drop_first', 'model': 'rf', 'rf_n_estimators': 161, 'rf_max_depth': 10, 'rf_min_samples_split': 20, 'rf_min_samples_leaf': 3}. Best is trial 101 with value: 0.8485263919579072.
[I 2026-05-08 12:05:25,767] Trial 113 pruned. 
[I 2026-05-08 12:05:27,223] Trial 114 finished with value: 0.8471885960750696 and parameters: {'scaler': 'robust', 'encoder': 'drop_first', 'model': 'rf', 'rf_n_estimators': 161, 'rf_max_depth': 10, 'rf_min_samples_split': 18, 'rf_min_samples_leaf': 4}. Best is trial 101 with value: 0.8485263919579

In [27]:
# =========================================================
# Results
# =========================================================

print("\nBest ROC-AUC:")
print(study.best_value)

print("\nBest Params:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")


Best ROC-AUC:
0.8485263919579072

Best Params:
scaler: robust
encoder: drop_first
model: rf
rf_n_estimators: 136
rf_max_depth: 10
rf_min_samples_split: 19
rf_min_samples_leaf: 3


In [22]:
importance = best_pipeline.named_steps['model'].feature_importances_

MLflow Experiment
    └── Optuna Study
            └── Trial Loop
                    ├── Choose hyperparameters (Bayesian optimization)
                    ├── Build preprocessing + model pipeline
                    ├── Run Cross Validation
                    ├── Compute ROC-AUC
                    ├── Report metric to Optuna
                    ├── Optuna decides:
                    │       ├── continue
                    │       └── prune trial
                    └── MLflow logs params + metrics
            └── Best trial selected
    └── Train best model on full training data
    └── Save model to MLflow

In [23]:
importance

array([0.00741789, 0.00806817, 0.00752347, 0.15420281, 0.08734166,
       0.014674  , 0.06466864, 0.01036639, 0.02261039, 0.1124184 ,
       0.07494955, 0.07545969, 0.28742476, 0.04327946, 0.0295947 ])